# Phase 5: Combined Model (Tabular + Graph Features)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

TABULAR_PATH = "/Users/shengqu/Desktop/CSCI5253/FinalProject/ieee-fraud-detection/data/train_with_uid.parquet"    
GRAPH_PATH   = "/Users/shengqu/Desktop/CSCI5253/FinalProject/ieee-fraud-detection/data/graph_features.parquet"    
MODEL_OUT    = "/Users/shengqu/Desktop/CSCI5253/FinalProject/model_output/xgb_combined.json"

## 1. Load & join

In [36]:
df = pd.read_parquet(TABULAR_PATH)
graph = pd.read_parquet(GRAPH_PATH)

df = df.merge(graph, on="TransactionID", how="left")
print(f"Combined shape: {df.shape}")
print(f"Graph feature columns added: {graph.shape[1] - 1}")

Combined shape: (590540, 472)
Graph feature columns added: 36


## 2. Time-based split

In [37]:
df = df.sort_values("TransactionDT").reset_index(drop=True)
split_idx = int(len(df) * 0.8)
train, valid = df.iloc[:split_idx], df.iloc[split_idx:]
print(f"Train: {len(train):,}  Valid: {len(valid):,}")

Train: 472,432  Valid: 118,108


## 3. Feature prep

In [ ]:
drop_cols = ["isFraud", "TransactionID", "TransactionDT",
             "uid", "card_age_day"]

y_train, y_valid = train["isFraud"], valid["isFraud"]
X_train = train.drop(columns=drop_cols).copy()
X_valid = valid.drop(columns=drop_cols).copy()

print(f"Starting feature count: {X_train.shape[1]}")

# Drop columns with >90% missing (based on train only)
miss = X_train.isnull().mean()
drop_high = miss[miss > 0.90].index.tolist()
X_train.drop(columns=drop_high, inplace=True)
X_valid.drop(columns=drop_high, inplace=True)
print(f"Dropped {len(drop_high)} columns with >90% missing.")

# Indicators for 50–90% missing
miss = X_train.isnull().mean()
mid = miss[(miss > 0.50) & (miss <= 0.90)].index.tolist()
for c in mid:
    X_train[f"{c}_missing"] = X_train[c].isnull().astype(np.int8)
    X_valid[f"{c}_missing"] = X_valid[c].isnull().astype(np.int8)
print(f"Added {len(mid)} missingness indicators.")

# Label-encode objects
for c in X_train.select_dtypes(include="object").columns:
    le = LabelEncoder()
    combined = pd.concat([X_train[c], X_valid[c]], axis=0).astype(str).fillna("NA")
    le.fit(combined)
    X_train[c] = le.transform(X_train[c].astype(str).fillna("NA"))
    X_valid[c] = le.transform(X_valid[c].astype(str).fillna("NA"))

print(f"Final feature count: {X_train.shape[1]}")

Starting feature count: 467
Dropped 12 columns with >90% missing.


/var/folders/ws/c4n8v3rs1_g18bkf6s3m1lhc0000gn/T/ipykernel_87142/2206865882.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train[f"{c}_missing"] = X_train[c].isnull().astype(np.int8)
/var/folders/ws/c4n8v3rs1_g18bkf6s3m1lhc0000gn/T/ipykernel_87142/2206865882.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_valid[f"{c}_missing"] = X_valid[c].isnull().astype(np.int8)
/var/folders/ws/c4n8v3rs1_g18bkf6s3m1lhc0000gn/T/ipykernel_87142/2206865882.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually 

Added 218 missingness indicators.
Final feature count: 673


## 4. Train

In [39]:
model = xgb.XGBClassifier(
    n_estimators=1000,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="auc",
    early_stopping_rounds=50,
    n_jobs=-1,
    random_state=42,
)
model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=50)

[0]	validation_0-auc:0.78245
[50]	validation_0-auc:0.88268
[82]	validation_0-auc:0.87977


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,50
,enable_categorical,False
,eval_metric,'auc'


## 5. Evaluate & compare to Phase 2

In [ ]:
p = model.predict_proba(X_valid)[:, 1]
roc = roc_auc_score(y_valid, p)
pr  = average_precision_score(y_valid, p)

print(f"Combined model (tabular + graph)")
print(f"  ROC-AUC : {roc:.4f}")
print(f"  PR-AUC  : {pr:.4f}   (naive baseline = {y_valid.mean():.4f})")
print()

Combined model (tabular + graph)
  ROC-AUC : 0.8853
  PR-AUC  : 0.4705   (naive baseline = 0.0344)

Compare to the Phase 2 baseline numbers you recorded.


### 5a. Precision / recall at fixed alert rates

In [41]:
for alert_rate in [0.005, 0.01, 0.02, 0.05]:
    threshold = np.quantile(p, 1 - alert_rate)
    flagged = p >= threshold
    precision = y_valid[flagged].mean()
    recall = flagged[y_valid == 1].mean() if y_valid.sum() else 0
    print(f"Alert rate {alert_rate:.1%}  precision={precision:.3f}  recall={recall:.3f}")

Alert rate 0.5%  precision=0.905  recall=0.132
Alert rate 1.0%  precision=0.840  recall=0.244
Alert rate 2.0%  precision=0.642  recall=0.373
Alert rate 5.0%  precision=0.356  recall=0.517


## 6. Feature importance

In [42]:
imp = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)

def tag(name):
    if name.endswith("_missing"): return "missing"
    if "__" in name:              return "graph"
    return "tabular"

imp_df = imp.rename("importance").to_frame()
imp_df["kind"] = imp_df.index.map(tag)
print("Top 30 features by gain:")
print(imp_df.head(30))

print("\nAggregate importance by kind:")
print(imp_df.groupby("kind")["importance"].sum().round(4))

Top 30 features by gain:
                            importance     kind
V246                          0.108038  tabular
V258                          0.105119  tabular
V295                          0.063542  tabular
V97                           0.038410  tabular
V189                          0.020477  tabular
card1__nbr_fraud_rate         0.018513    graph
V156                          0.017202  tabular
V317                          0.016729  tabular
C7                            0.014703  tabular
V257                          0.014261  tabular
V70                           0.014037  tabular
V294                          0.011714  tabular
C14                           0.010193  tabular
V308                          0.010093  tabular
M5_missing                    0.009177  missing
V91                           0.008911  tabular
V187                          0.008226  tabular
C1                            0.008190  tabular
V102                          0.007632  tabular
DeviceInfo__nbr

## 7. Save

In [ ]:
model.save_model(MODEL_OUT)
print(f"Saved combined model to {MODEL_OUT}")

Saved combined model to /Users/shengqu/Desktop/CSCI5253/FinalProject/model_output/xgb_combined.json
Next step: 06_export_artifact.ipynb — bundle the model + graph lookup tables for the FastAPI worker.


In [44]:
# Is it the extra features or the fewer rounds?
print("best_iteration:", model.best_iteration)

# How much are graph features contributing?
imp = pd.Series(model.feature_importances_, index=X_train.columns)
imp_df = imp.rename("importance").to_frame()
imp_df["kind"] = imp_df.index.map(lambda n: "graph" if "__" in n else "missing" if n.endswith("_missing") else "tabular")
print(imp_df.groupby("kind")["importance"].sum())

# Train a model WITH ONLY the graph features — are they informative at all?
graph_cols = [c for c in X_train.columns if "__" in c]
gm = xgb.XGBClassifier(**model.get_params())
gm.fit(X_train[graph_cols], y_train, eval_set=[(X_valid[graph_cols], y_valid)], verbose=False)
p = gm.predict_proba(X_valid[graph_cols])[:, 1]
print("Graph-features-only ROC-AUC:", roc_auc_score(y_valid, p))

best_iteration: 32
kind
graph      0.065324
missing    0.017608
tabular    0.917067
Name: importance, dtype: float32
Graph-features-only ROC-AUC: 0.7969081851903685
